In [47]:
!pip install tensorflow pandas numpy scikit-learn tensorboard matplotlib streamlit scikeras nltk

In [48]:
import nltk
nltk.download("gutenberg")
from nltk.corpus import gutenberg
import pandas as pd

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [49]:
## load data
data = gutenberg.raw('shakespeare-hamlet.txt')

In [50]:
## save to a file
with open('hamlet.txt', 'w') as file:
  file.write(data)

In [51]:
## data preprocessing

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer                         # helps in converting text into vectors
from tensorflow.keras.preprocessing.sequence import pad_sequences                 # it will makesure that all the length will be same
from sklearn.model_selection import train_test_split

In [52]:
# Load the dataset

with open ('hamlet.txt', 'r') as file:
  text = file.read().lower()

# tokenize the text

token = Tokenizer()     # object of the class

token.fit_on_texts([text])                                    # creating the indexes for the words
total_words = len(token.word_index) + 1
total_words


4818

In [53]:
# create input sequence

inputSequence = []
for line in text.split('\n'):
  token_list = token.texts_to_sequences([line])[0]
  for i in range(1, len(token_list)):
    n_gram_sequence = token_list[:i + 1]
    inputSequence.append(n_gram_sequence)


In [54]:
## pad sequence

max_sequence_length = max([len(x) for x in inputSequence])
max_sequence_length

14

In [55]:
inputSequence = np.array(pad_sequences(inputSequence, maxlen=max_sequence_length, padding='pre'))


In [56]:
## create a predictors and label

import tensorflow as tf
x, y = inputSequence[:, :-1], inputSequence[:, -1]                # all word except the last will go to the x and last word will go to the y

x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]], dtype=int32)

In [57]:
y = tf.keras.utils.to_categorical(y, num_classes=total_words)          # converts the output feature to the categories, all will be zero except one number
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [58]:
# Split the data into the train, test split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [59]:
# # Applying the earlystoping

# from tensorflow.keras.callbacks import EarlyStopping
# early_stopping = EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights = True)



In [60]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dropout, Dense

model = Sequential([
    Input(shape=(max_sequence_length - 1,)),
    Embedding(input_dim=total_words, output_dim=100),
    LSTM(150),  # No return_sequences=True
    Dropout(0.2),
    Dense(total_words, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

print("Output shape:", model.output_shape)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 150)            │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4818)           │       727,518 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,359,918 (5.19 MB)

 Trainable params: 1,359,918 (5.19 MB)

 Non-trainable params: 0 (0.00 B)

Output shape: (None, 4818)


In [61]:
print("Current model output:", model.output_shape)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

Current model output: (None, 4818)
X_train shape: (20585, 13)
y_train shape: (20585, 4818)


In [62]:
## train our model

# Using X_test since it was named X_test during train_test_split

history = model.fit(X_train, y_train, epochs= 50, validation_data= (X_test, y_test), verbose=1)

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.0353 - loss: 6.8198 - val_accuracy: 0.0396 - val_loss: 6.6806
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.0432 - loss: 6.4240 - val_accuracy: 0.0476 - val_loss: 6.7077
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.0553 - loss: 6.2287 - val_accuracy: 0.0538 - val_loss: 6.7643
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.0643 - loss: 6.0356 - val_accuracy: 0.0624 - val_loss: 6.8175
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.0765 - loss: 5.8145 - val_accuracy: 0.0688 - val_loss: 6.8897
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.0888 - loss: 5.5768 - val_accuracy: 0.0746 - val_loss: 6.9801
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.0979 - loss: 5.3338 - val_accuracy: 0.0723 - val_loss: 7.0598
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.1114 - loss: 5.0808 - val_accuracy

In [63]:
# Function to predict the next words

def predict_next_words(model, token, text, max_sequence_length):
  token_list = token.texts_to_sequences([text])[0]
  if len(token_list) >= max_sequence_length:
    token_list = token_list[-(max_sequence_length - 1):]            #Ensure the Sequence Length matches the max_sequence
  token_list = pad_sequences([token_list], maxlen=max_sequence_length - 1, padding='pre')
  predicted = model.predict(token_list, verbose=0)
  predicted_word_index = np.argmax(predicted, axis=1)
  for word, index in token.word_index.items():
    if index == predicted_word_index:
      return word
  return None

In [65]:
input_text = "To be or not to be"
print(f"Input Text:{input_text}")
max_sequence_length = model.input_shape[1] + 1
next_word = predict_next_words(model, token, input_text, max_sequence_length)
print(f"Next Word Prediction: {next_word}")


Input Text:To be or not to be
Next Word Prediction: all


In [70]:
! pip install pickle

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


In [72]:
from typing import Protocol
# Save the model

model.save("next_word_lstm.h5")

# Save the tokenizer
import pickle
with open("tokenizer.pkl", "wb") as handle:
  pickle.dump(token, handle, protocol = pickle.HIGHEST_PROTOCOL)
